# Visual ML Foundations: Mass Spectrometry Peak Processing
### Centroiding, Noise Thresholding, and m/z Spectral Alignment

---

## 1. Concept Overview
In natural product and metabolomic discovery (like the **Enveda CASMI** challenge), molecules are identified by ionizing them and measuring the mass-to-charge ratio ($m/z$) of their fragment ions.
Raw mass spectra contain thousands of raw detector noise spikes alongside true chemical adducts. Preprocessing requires:
1. **Intensity Thresholding**: Eliminating electronic baseline noise.
2. **Peak Centroiding**: Converting continuous mass peaks into discrete $(m/z, 	ext{intensity})$ tuples.
3. **Square Root / Log Transformation**: Compressing intense precursor peaks so subtle fragment ions remain informative.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generate simulated mass spectrum with precursor and fragment peaks
np.random.seed(42)
mz_axis = np.linspace(50, 600, 2000)
raw_spectrum = np.random.exponential(scale=2.0, size=len(mz_axis))

# Known chemical ion fragments (m/z, intensity)
true_peaks = [(105.04, 85.0), (149.02, 60.0), (279.16, 95.0), (391.28, 100.0), (413.26, 45.0)]
for mz, intensity in true_peaks:
    idx = np.argmin(np.abs(mz_axis - mz))
    raw_spectrum[idx-3:idx+4] += intensity * np.exp(-0.5 * (np.linspace(-2, 2, 7))**2)

# Apply noise thresholding & square root transformation
threshold = 10.0
clean_spectrum = np.where(raw_spectrum > threshold, raw_spectrum, 0.0)
normalized_spectrum = np.sqrt(clean_spectrum)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax1.plot(mz_axis, raw_spectrum, color='crimson', lw=1)
ax1.axhline(threshold, color='black', linestyle='--', label=f'Noise Gate ({threshold})')
ax1.set_title("Raw Mass Spectrum with Baseline Noise", fontsize=12, fontweight='bold')
ax1.set_ylabel("Intensity")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.vlines(mz_axis[clean_spectrum > 0], 0, normalized_spectrum[clean_spectrum > 0], color='navy', lw=1.5)
ax2.set_title("Centroided & Sqrt-Normalized Feature Spectrum", fontsize=12, fontweight='bold')
ax2.set_xlabel("Mass-to-charge ratio (m/z)")
ax2.set_ylabel("Normalized Intensity")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Extracted {np.sum(clean_spectrum > 0)} significant spectral feature peaks.")
